In [38]:
# Imports
import os
import sys
import json
import math
import cv2
import base64
import io
from datetime import datetime
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import SpectralClustering
from pydantic import BaseModel
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from cosette import *
from toolslm.md_hier import *
from collections import defaultdict
from hashlib import sha256

In [8]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Chris Sweet, Priscila Moreira
Date Created: February 23, 2025
Description:  Paragraph merging code to keep key-pairs in the same cluster. 
              The new merged paragraphs are appended directly into the 
              `"paragraphs"` list. The original paragraphs that contributed to 
              a merge are marked with `"invalid": true` and a `"mergedInto"` 
              pointer, so you can later filter or review them if needed.
--------------------------------------------------------------------------------
"""

def get_max_paragraph_id(ocr_json):
    """
    Return the maximum paragraph id among the paragraphs.
    """
    ids = []
    for para in ocr_json.get("paragraphs", []):
        try:
            ids.append(int(para.get("id")))
        except Exception:
            continue
    #print("ids", ids)
    return max(ids) if ids else 1000

def polygon_to_bbox(polygon):
    """
    Convert a flat polygon array into a bounding box [x_min, y_min, x_max, y_max].
    Assumes polygon is a flat list of numbers: [x1, y1, x2, y2, ...].
    """
    pts = np.array(polygon).reshape(-1, 2)
    x_min = float(pts[:, 0].min())
    y_min = float(pts[:, 1].min())
    x_max = float(pts[:, 0].max())
    y_max = float(pts[:, 1].max())
    return [x_min, y_min, x_max, y_max]

def bbox_to_polygon(bbox):
    """
    Convert a bounding box [x_min, y_min, x_max, y_max] into a flattened polygon array 
    with four points: top-left, top-right, bottom-right, bottom-left.
    """
    x_min, y_min, x_max, y_max = bbox
    return [x_min, y_min, x_max, y_min, x_max, y_max, x_min, y_max]

def union_bbox(bbox_list):
    """
    Given a list of bounding boxes [x_min, y_min, x_max, y_max],
    return the union bounding box that encloses them all.
    """
    if not bbox_list:
        return None
    xmins = [b[0] for b in bbox_list]
    ymins = [b[1] for b in bbox_list]
    xmaxs = [b[2] for b in bbox_list]
    ymaxs = [b[3] for b in bbox_list]
    return [min(xmins), min(ymins), max(xmaxs), max(ymaxs)]

def compute_iou(bbox1, bbox2):
    """Compute Intersection over Union (IoU) for two bounding boxes."""
    xA = max(bbox1[0], bbox2[0])
    yA = max(bbox1[1], bbox2[1])
    xB = min(bbox1[2], bbox2[2])
    yB = min(bbox1[3], bbox2[3])
    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH
    area1 = (bbox1[2]-bbox1[0]) * (bbox1[3]-bbox1[1])
    area2 = (bbox2[2]-bbox2[0]) * (bbox2[3]-bbox2[1])
    unionArea = area1 + area2 - interArea
    return interArea / unionArea if unionArea else 0

def get_paragraph_bbox(para):
    """
    Compute the overall bounding box for a paragraph by unioning its boundingRegions.
    """
    bboxes = []
    for region in para.get("boundingRegions", []):
        poly = region.get("polygon", [])
        if poly:
            bboxes.append(polygon_to_bbox(poly))
    if bboxes:
        return union_bbox(bboxes)
    return None

def create_merged_paragraph(key_para, value_para):
    """
    Create a new merged paragraph element.
    The new paragraph element will:
      - Concatenate key and value content.
      - Compute a merged bounding box (union of key and value bboxes),
        then convert that bbox into a full polygon with 8 entries.
      - Create a new span.
      - Record contributing paragraph IDs.
    """
    key_text = key_para.get("content", "").strip()
    value_text = value_para.get("content", "").strip()
    merged_text = key_text + " " + value_text if value_text not in key_text else key_text

    key_bbox = get_paragraph_bbox(key_para)
    value_bbox = get_paragraph_bbox(value_para)
    merged_bbox = union_bbox([key_bbox, value_bbox])
    merged_poly = bbox_to_polygon(merged_bbox) if merged_bbox else []

    # Copy pageNumber from key paragraph's first bounding region if available.
    page_number = None
    if key_para.get("boundingRegions"):
        page_number = key_para["boundingRegions"][0].get("pageNumber")

    merged_region = {
        "pageNumber": page_number,
        "polygon": merged_poly
    }
    
    new_span = {"offset": 0, "length": len(merged_text)}
    
    merged_para = {
        "content": merged_text,
        "boundingRegions": [merged_region],
        "spans": [new_span],
        "contributingParagraphs": [key_para.get("id", None), value_para.get("id", None)]
    }
    return merged_para

def merge_key_value_pairs_create_new_paragraphs(ocr_json, iou_threshold=0.1):
    """
    Process each keyValuePair in the OCR JSON:
      - Identify the best matching key paragraph and value paragraph.
      - Create a new merged paragraph element.
      - Append the new merged paragraph into the OCR JSON's "paragraphs" list.
      - Mark the original key and value paragraphs with "invalid": True and "mergedInto": <merged_id>.
    """
    paragraphs = ocr_json.get("paragraphs", [])
    
    # Ensure each paragraph has an "id"
    for idx, para in enumerate(paragraphs):
        if "id" not in para:
            para["id"] = idx

    new_merged_paragraphs = []
    merged_paragraphs_current_id = get_max_paragraph_id(ocr_json) + 1

    for kv in ocr_json.get("keyValuePairs", []):
        key_obj = kv.get("key", {})
        value_obj = kv.get("value", {})
        if not key_obj or not value_obj:
            continue

        key_bboxes = [polygon_to_bbox(r.get("polygon", []))
                      for r in key_obj.get("boundingRegions", [])
                      if r.get("polygon")]
        value_bboxes = [polygon_to_bbox(r.get("polygon", []))
                        for r in value_obj.get("boundingRegions", [])
                        if r.get("polygon")]
        if not key_bboxes or not value_bboxes:
            continue

        key_bbox_union = union_bbox(key_bboxes)
        value_bbox_union = union_bbox(value_bboxes)

        best_key_iou = 0
        best_key_idx = None
        best_value_iou = 0
        best_value_idx = None

        for idx, para in enumerate(paragraphs):
            para_bbox = get_paragraph_bbox(para)
            if para_bbox is None:
                continue
            iou_key = compute_iou(key_bbox_union, para_bbox)
            iou_value = compute_iou(value_bbox_union, para_bbox)
            if iou_key > best_key_iou:
                best_key_iou = iou_key
                best_key_idx = idx
            if iou_value > best_value_iou:
                best_value_iou = iou_value
                best_value_idx = idx

        if (best_key_idx is not None and best_value_idx is not None and 
            best_key_idx != best_value_idx and 
            best_key_iou >= iou_threshold and best_value_iou >= iou_threshold):
            
            key_para = paragraphs[best_key_idx]
            value_para = paragraphs[best_value_idx]

            merged_para = create_merged_paragraph(key_para, value_para)
            # Assign a new ID to the merged paragraph.
            merged_para["id"] = merged_paragraphs_current_id #f"merged_{key_para.get('id')}_{value_para.get('id')}"
            # add key value key information
            merged_para["key_value_pairs"] = [{"key":key_obj["content"], "value": value_obj["content"]}]

            merged_paragraphs_current_id += 1
            new_merged_paragraphs.append(merged_para)
            
            # Mark original paragraphs as merged.
            key_para["invalid"] = True
            key_para["mergedInto"] = merged_para["id"]

            value_para["invalid"] = True
            value_para["mergedInto"] = merged_para["id"]


    # Append new merged paragraphs to the existing paragraphs list.
    ocr_json["paragraphs"].extend(new_merged_paragraphs)
    return ocr_json


In [9]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Chris Sweet, Priscila Moreira
Date Created: February 23, 2025
Description:  Paragraph merging code to keep tables in the same cluster. 
              The new merged paragraphs are appended directly into the 
              `"paragraphs"` list. The original paragraphs that contributed to 
              a merge are marked with `"invalid": true` and a `"mergedInto"` 
              pointer, so you can later filter or review them if needed.
--------------------------------------------------------------------------------
"""

def get_table_csv(table):
    """
    Convert a table's cell contents into a CSV string.
    Uses the "rowIndex" and "columnIndex" to build a grid.
    """
    rows = table.get("rowCount", 0)
    cols = table.get("columnCount", 0)
    # Initialize grid with empty strings.
    grid = [["" for _ in range(cols)] for _ in range(rows)]
    
    for cell in table.get("cells", []):
        row = cell.get("rowIndex", 0)
        col = cell.get("columnIndex", 0)
        text = cell.get("content", "").strip()
        grid[row][col] = text

    # Create CSV lines by joining each row with commas.
    csv_lines = [",".join(row) for row in grid]
    return "\n".join(csv_lines)

def find_paragraph_by_pointer(ocr_json, pointer):
    """
    Given a pointer string (e.g., "/paragraphs/6"), return the corresponding paragraph object
    from ocr_json["paragraphs"]. Assumes pointer format is fixed.
    """
    try:
        _, _, id_str = pointer.partition("/paragraphs/")
        target_id = int(id_str)
    except Exception:
        return None

    for para in ocr_json.get("paragraphs", []):
        try:
            if int(para.get("id", -1)) == target_id:
                return para
        except Exception:
            continue
    return None

def get_max_paragraph_id(ocr_json):
    """
    Return the maximum paragraph id among the paragraphs.
    """
    ids = []
    for para in ocr_json.get("paragraphs", []):
        try:
            ids.append(int(para.get("id")))
        except Exception:
            continue
    return max(ids) if ids else 0

def merge_table_to_paragraph(ocr_json):
    """
    For each table in ocr_json["tables"]:
      - Convert the table cells to CSV text.
      - Use the table's own "boundingRegions" as the new paragraph's boundingRegions.
      - For each cell that references a paragraph via "elements", mark that paragraph as invalid and set "mergedInto".
      - Create a new paragraph element with:
          * "content": the CSV text,
          * "boundingRegions": the table's boundingRegions,
          * "spans": a new span covering the CSV text,
          * "contributingParagraphs": a list of paragraph ids referenced by the table.
      - Append the new paragraph to ocr_json["paragraphs"].
    """
    paragraphs = ocr_json.get("paragraphs", [])
    
    # Ensure each paragraph has an "id"
    for idx, para in enumerate(paragraphs):
        if "id" not in para:
            para["id"] = idx

    max_id = get_max_paragraph_id(ocr_json)
    new_paragraphs = []

    for table in ocr_json.get("tables", []):
        csv_text = get_table_csv(table)
        
        # Use the table's boundingRegions directly.
        table_regions = table.get("boundingRegions", [])
        if not table_regions:
            continue
        
        # Collect contributing paragraph ids from the table's cells.
        contributing_ids = set()
        for cell in table.get("cells", []):
            for pointer in cell.get("elements", []):
                para_obj = find_paragraph_by_pointer(ocr_json, pointer)
                if para_obj and "id" in para_obj:
                    contributing_ids.add(int(para_obj["id"]))
        
        # Create a new paragraph element for the merged table.
        new_id = max_id + 1
        max_id += 1
        new_para = {
            "id": new_id,
            "content": csv_text,
            "boundingRegions": table_regions,  # Directly use table's boundingRegions.
            "spans": [{"offset": 0, "length": len(csv_text)}],
            "contributingParagraphs": list(contributing_ids)
        }
        
        # Mark the contributing paragraphs as invalid and note their mergedInto.
        # and collect any key pairs
        key_pairs = []
        key_pairs_processed = []

        for pid in contributing_ids:
            for para in paragraphs:
                try:
                    if int(para.get("id")) == pid:
                        para["invalid"] = True

                        # if previously merged we need to invalidate the merged to paragraph
                        merged_to_p = int(para.get("mergedInto", -1))

                        if merged_to_p != -1:
                            # find the para with that id
                            result_p = next((para for para in ocr_data["paragraphs"] if int(para.get("id", -1)) == merged_to_p), None)
                            if result_p:
                                #print("Found paragraph:", result_p)
                                # set it invalid
                                result_p["invalid"] = True

                                # add in any key pairs in the merged contributed paragraphs
                                key_pairs_p = result_p.get("key_value_pairs", None)
                                if key_pairs_p:
                                    if merged_to_p not in key_pairs_processed:
                                        key_pairs.append(key_pairs_p)
                                        key_pairs_processed.append(merged_to_p)

                            # else:
                            #     print("Paragraph not found.")
                        
                        # point each paragraph to our new addition
                        para["mergedInto"] = new_id

                        # add in any key pairs in the contributed paragraphs
                        key_pairs_p = para.get("key_value_pairs", None)
                        if key_pairs_p:
                            key_pairs.append(key_pairs_p)
    
                except Exception:
                    continue

        # add key pairs to table merge paragraph
        if key_pairs:
            new_para["key_value_pairs"] = key_pairs
        new_paragraphs.append(new_para)
        #print(f"New paragraph {json.dumps(new_para)}")

    # Append new merged paragraphs into the existing paragraphs list.
    ocr_json["paragraphs"].extend(new_paragraphs)
    return ocr_json


In [10]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Chris Sweet, Priscila Moreira
Date Created: February 23, 2025
Description: The paragraph clustering code combines text-embedding similarity 
             (from Sentence-BERT) with spatial proximity (derived from bounding 
             box centers) into a composite similarity matrix. In this example, 
             we compute the cosine similarity between paragraph embeddings 
             and a spatial similarity score (using a Gaussian kernel on the 
             Euclidean distance between bounding box centers). We then weight 
             these two scores (using a parameter alpha) and run spectral 
             clustering with the combined affinity matrix.
--------------------------------------------------------------------------------
"""

def extract_paragraphs(ocr_data):
    """
    Extract paragraph texts and bounding boxes.
    Here, we assume each paragraph has a "content" string and a list of "boundingRegions".
    We take the first bounding region and compute its center.
    Returns lists of texts and centers (as [x_center, y_center]).
    """
    paragraphs = ocr_data.get("paragraphs", [])
    texts = []
    centers = []
    ids = []
    
    for para in paragraphs:
        # check if valid
        if para.get("invalid", False):
            #print("Invalid id", str(para.get("id", -1)))
            continue
        # else:
        #     print("Valid id", str(para.get("id", -1)))

        # check if it has a role
        if para.get("role", "") != "":
            #print("Paragraph has role", str(para.get("role", "")))
            continue

        text = para.get("content", "").strip()
        if not text:
            continue

        # Use the first bounding region to compute the center.
        brs = para.get("boundingRegions", [])
        if not brs:
            continue
        # Each bounding region has a "polygon" of points: [x1, y1, x2, y2, ..., xN, yN]
        # We'll assume the polygon is rectangular and compute center from min/max.
        poly = brs[0].get("polygon", [])
        if not poly or len(poly) < 4:
            continue

        # save paragraph id
        ids.append(para.get("id", -1))

        # Convert polygon list into pairs:
        pts = np.array(poly).reshape(-1, 2)
        x_min, y_min = pts.min(axis=0)
        x_max, y_max = pts.max(axis=0)
        center = [(x_min + x_max) / 2, (y_min + y_max) / 2]

        texts.append(text)
        centers.append(center)
    return ids, texts, np.array(centers)

def compute_spatial_similarity(centers, sigma=50.0):
    """
    Compute spatial similarity matrix given the centers of paragraphs.
    We use a Gaussian kernel: sim(i, j) = exp(-||center_i - center_j||^2 / (2 * sigma^2))
    sigma should be set according to the scale of your coordinates.
    """
    num = centers.shape[0]
    sim = np.zeros((num, num))
    for i in range(num):
        for j in range(num):
            dist = np.linalg.norm(centers[i] - centers[j])
            sim[i, j] = np.exp(- (dist ** 2) / (2 * sigma ** 2))
    return sim

def compute_text_similarity(texts, model):
    """Compute cosine similarity between paragraph embeddings."""
    embeddings = model.encode(texts, show_progress_bar=True)
    return cosine_similarity(embeddings), embeddings

def composite_similarity(text_sim, spatial_sim, alpha=0.7):
    """
    Combine text and spatial similarities.
    alpha: weight for text similarity, (1-alpha) for spatial.
    """
    return alpha * text_sim + (1 - alpha) * spatial_sim

def cluster_paragraphs(affinity, n_clusters=5):
    """
    Cluster paragraphs using Spectral Clustering with a precomputed affinity matrix.
    """
    sc = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', random_state=42)
    return sc.fit_predict(affinity)

def generate_clusters(ocr_data):
    # Extract paragraph texts and bounding box centers
    ids, texts, centers = extract_paragraphs(ocr_data)
    if not texts:
        #print("No paragraphs found!")
        return None

    # Load Sentence-BERT model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Compute text similarity matrix
    text_sim, embeddings = compute_text_similarity(texts, model)

    # Compute spatial similarity matrix (sigma is a tuning parameter)
    spatial_sim = compute_spatial_similarity(centers, sigma=100.0)

    # Combine similarities: alpha controls weighting (0 <= alpha <= 1)
    alpha = 0.3  # Higher weight for text similarity
    affinity = composite_similarity(text_sim, spatial_sim, alpha=alpha)

    # Cluster paragraphs using the composite similarity matrix.
    n_clusters = 8  # Adjust as needed based on your document
    labels = cluster_paragraphs(affinity, n_clusters=n_clusters)

    # Assuming ids, texts, centers, and labels have been defined previously
    cluster_labels = {}
    for pid, text, center, label in zip(ids, texts, centers, labels):
        # Convert label to a standard Python value (if it's a tensor, for example)
        label_val = label.item() if hasattr(label, "item") else label
        cluster_labels[pid] = {"label": label_val, "name": f"C{label_val}", "type": "content"}
        # print(f"Paragraph {pid} (Cluster {label_val}, Center: {center}):")
        # print(text)
        # print("-" * 80)

    # grab the paragraphs with roles
    paragraph_roles = {
        para["id"]: para["role"]
        for para in ocr_data.get("paragraphs", [])
        if para.get("role", "").strip() != ""
    }

    # add to the cluster
    pr_count = n_clusters

    # loop
    for para_id, role in paragraph_roles.items():
        #print(f"Paragraph ID: {para_id}, Role: {role}")
        cluster_labels[para_id] = {"label": pr_count, "name": f"C{pr_count}", "type": role}
        pr_count += 1

    #print("Cluster Label Dictionary:", cluster_labels)
    return cluster_labels


In [11]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Chris Sweet, Priscila Moreira
Date Created: February 23, 2025
Description: The code groups paragraphs by cluster label and then sorts the 
             paragraphs within each cluster by their vertical position (using 
             the top coordinate of their bounding box). Finally, it aggregates 
             the text for each cluster in the correct reading order.
--------------------------------------------------------------------------------
"""

def polygon_to_bbox(polygon):
    """
    Convert a flat polygon array into a bounding box [x_min, y_min, x_max, y_max].
    Assumes polygon is a flat list of numbers: [x1, y1, x2, y2, ...].
    """
    pts = np.array(polygon).reshape(-1, 2)
    x_min = float(pts[:, 0].min())
    y_min = float(pts[:, 1].min())
    x_max = float(pts[:, 0].max())
    y_max = float(pts[:, 1].max())
    return [x_min, y_min, x_max, y_max]

def union_bbox(bbox_list):
    """
    Given a list of bounding boxes [x_min, y_min, x_max, y_max],
    return the union bounding box that encloses them all.
    """
    if not bbox_list:
        return None
    xmins = [bbox[0] for bbox in bbox_list]
    ymins = [bbox[1] for bbox in bbox_list]
    xmaxs = [bbox[2] for bbox in bbox_list]
    ymaxs = [bbox[3] for bbox in bbox_list]
    return [min(xmins), min(ymins), max(xmaxs), max(ymaxs)]

def get_paragraph_top(para):
    """
    Return the top (y_min) coordinate from the first bounding region of a paragraph.
    If no bounding region exists, return infinity.
    """
    regions = para.get("boundingRegions", [])
    if not regions:
        return float('inf')
    poly = regions[0].get("polygon", [])
    if not poly:
        return float('inf')
    return polygon_to_bbox(poly)[1]

def group_paragraphs_by_cluster(ocr_data, labels_dict):
    """
    Group paragraphs by cluster and extract the cluster name at the same time.
    
    Parameters:
      paragraphs: List of paragraph objects from OCR JSON.
      labels_dict: Dictionary mapping paragraph id to label info.
    
    Returns:
      A dictionary where keys are cluster labels and values are dicts containing:
         - "name": the cluster name (from labels_dict)
         - "text": concatenated paragraph text in reading order
         - "boundingBox": union bounding box of all paragraphs in the cluster.
    """

    # Get paragraphs from ocr
    paragraphs = ocr_data.get("paragraphs", [])
    
    # Create a dictionary mapping cluster labels to {"name": <name>, "paragraphs": [...]}
    cluster_dict = {}
    for para in paragraphs:
        pid = para.get("id", -1)
        if pid in labels_dict:
            label = labels_dict[pid]["label"]
            name = labels_dict[pid]["name"]
            c_type = labels_dict[pid]["type"]
            cluster_dict.setdefault(label, {"name": name, "paragraphs": [], "type": c_type})
            cluster_dict[label]["paragraphs"].append(para)
    
    # Process each cluster to compute combined text and union bounding box.
    cluster_output = {}
    for lbl, data in cluster_dict.items():
        paras = data["paragraphs"]
        # Sort paragraphs by vertical position, using top coordinate, and x_min as tiebreaker.
        paras_sorted = sorted(
            paras,
            key=lambda p: (
                get_paragraph_top(p),
                polygon_to_bbox(p.get("boundingRegions", [])[0]["polygon"]) if p.get("boundingRegions", []) else [0,0,0,0]
            )
        )
        combined_text = "\n".join(p.get("content", "").strip() for p in paras_sorted)
        # Compute union bounding box using the first bounding region from each paragraph.
        bboxes = []
        for p in paras_sorted:
            if p.get("boundingRegions"):
                bboxes.append(polygon_to_bbox(p["boundingRegions"][0]["polygon"]))
        union_box = union_bbox(bboxes) if bboxes else None
        
        # get key value pairs
        key_pairs = []

        for p in paras_sorted:
            # add in any key pairs in the contributed paragraphs
            key_pairs_p = p.get("key_value_pairs", None)
            if key_pairs_p:
                #key_pairs.extend(key_pairs_p)
                # If key_pairs_p is a list of dicts: extend directly
                # If key_pairs_p is a list containing sub-lists: flatten
                for item in key_pairs_p:
                    if isinstance(item, list):
                        # Flatten the sub-list
                        key_pairs.extend(item)
                    else:
                        # It's already a dict or object, just append
                        key_pairs.append(item)

        cluster_output[lbl] = {
            "name": data["name"],
            "text": combined_text,
            "boundingBox": union_box,
            "key_value_pairs": key_pairs,
            "type": data["type"]
        }
    return cluster_output


In [12]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Chris Sweet, Priscila Moreira
Date Created: February 23, 2025
Description: Finds positional relationships between clusters within a given 
             margin and finds the reading order of the clusters.
--------------------------------------------------------------------------------
"""

def bbox_relationship(b1, b2, margin):
    """
    Return a list of positional relationships between bounding boxes b1 and b2,
    with a margin/cutoff. Each bounding box is [x_min, y_min, x_max, y_max].
    
    Possible relationships: "inside", "contains", "left", "right", "above", "below", "overlaps".
    
    margin: the distance in pixels/coordinate units to use as a buffer.
    """
    x1_min, y1_min, x1_max, y1_max = b1
    x2_min, y2_min, x2_max, y2_max = b2
    
    relationships = []
    
    # Check inside
    if (x1_min >= x2_min and y1_min >= y2_min and 
        x1_max <= x2_max and y1_max <= y2_max):
        relationships.append("inside")
    
    # Check contains
    if (x2_min >= x1_min and y2_min >= y1_min and 
        x2_max <= x1_max and y2_max <= y1_max):
        relationships.append("contains")
    
    # Check left/right with margin
    # Now b1 is "left" of b2 only if b1.x_max + margin < b2.x_min
    if x1_max + margin < x2_min:
        relationships.append("left")
    
    # b1 is "right" of b2 only if b1.x_min - margin > b2.x_max
    if x1_min - margin > x2_max:
        relationships.append("right")
    
    # Check above/below with margin
    # b1 is "above" b2 if b1.y_max + margin < b2.y_min
    if y1_max + margin < y2_min:
        relationships.append("above")
    
    # b1 is "below" b2 if b1.y_min - margin > b2.y_max
    if y1_min - margin > y2_max:
        relationships.append("below")
    
    # If none of these strict separation conditions applied, we can check "overlaps"
    # or skip if not needed. We'll do a simple bounding box overlap check:
    if not any(r in ["left","right","above","below","contains","inside"] for r in relationships):
        overlap_width = min(x1_max, x2_max) - max(x1_min, x2_min)
        overlap_height = min(y1_max, y2_max) - max(y1_min, y2_min)
        if overlap_width > 0 and overlap_height > 0:
            relationships.append("overlaps")
    
    return relationships

def compute_cluster_relationships(clusters, margin=10):
    """
    clusters is a dict { cluster_id: { "boundingBox": [x_min,y_min,x_max,y_max], ... }, ... }
    Return a dict of form { (cid1, cid2): [list_of_relationships], ... }
    """
    cluster_ids = sorted(clusters.keys())
    rels = {}
    
    for i in range(len(cluster_ids)):
        for j in range(i+1, len(cluster_ids)):
            cid1 = cluster_ids[i]
            cid2 = cluster_ids[j]
            
            b1 = clusters[cid1].get("boundingBox")
            b2 = clusters[cid2].get("boundingBox")
            if b1 and b2:
                r12 = bbox_relationship(b1, b2, margin)
                r21 = bbox_relationship(b2, b1, margin)  # symmetrical but can differ on inside/contains
                rels[(cid1, cid2)] = r12
                rels[(cid2, cid1)] = r21
    
    return rels

def transform_relationships(rel_dict, clusters):
    """
    Convert pairwise relationship dictionary into a dict mapping each cluster ID
    to a list of relationship objects like:
        [ { "type": "above", "target": <other_cluster_id> }, ... ].
    
    rel_dict is in the format:  { (cid1, cid2): [relations...], ... }
    """
    cluster_relationships = defaultdict(list)
    
    # Loop over each pair (cid1, cid2) -> [list_of_relations]
    for (cid1, cid2), rels in rel_dict.items():
        for rel in rels:
            # Append { "type": rel, "target": cid2 } to the cluster cid1
            cluster_relationships[cid1].append({"type": rel, "target": clusters[cid2]["name"]})
    
    return dict(cluster_relationships)  # Convert defaultdict back to a normal dict

def sort_clusters_reading_order(clusters):
    """
    Sorts clusters in a reading order: top to bottom, then left to right.
    Clusters is a dict keyed by cluster_id, each with "boundingBox": [x_min,y_min,x_max,y_max].
    
    Returns a list of (cluster_id, cluster_info) in sorted order.
    """
    # Convert clusters dict items to a list for sorting
    cluster_items = list(clusters.items())
    
    def reading_key(item):
        cid, cinfo = item
        bbox = cinfo.get("boundingBox", [0,0,0,0])
        x_min, y_min, x_max, y_max = bbox
        return (y_min, x_min)  # Sort by top edge, then left edge
    
    # Sort by y_min ascending, then x_min ascending
    cluster_sorted = sorted(cluster_items, key=reading_key)
    return cluster_sorted

def classify_position(bbox, page_width, page_height):
    """
    Classify the position of a cluster's bounding box on a page.
    bbox: [x_min, y_min, x_max, y_max]
    page_width, page_height: dimensions of the page
    Returns a string label like "top of page", "below header", "middle of page",
    "bottom of page", "right hand side middle", etc.
    
    Adjust thresholds as needed for your layout definitions.
    """
    x_min, y_min, x_max, y_max = bbox
    
    # Compute center
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    
    # Convert to relative coordinates [0..1]
    rel_x = center_x / page_width
    rel_y = center_y / page_height
    
    # Simple thresholds (you can refine these):
    # For vertical position:
    if rel_y < 0.15:
        vertical_label = "top of page"
    elif rel_y < 0.35:
        vertical_label = "below header"
    elif rel_y < 0.7:
        vertical_label = "middle of page"
    else:
        vertical_label = "bottom of page"
    
    # For horizontal:
    # If center is on the right half, check if it's "right hand side middle"
    # or maybe just "right side"
    if rel_x > 0.5:
        horizontal_label = "right side"
    else:
        horizontal_label = "left side"

    # Combine logic to produce final label
    # E.g. "top of page" or "right hand side middle" if both conditions match
    # For instance:
    if vertical_label == "middle of page" and horizontal_label == "right side":
        return "right hand side middle"
    else:
        # Return the vertical_label as primary
        return vertical_label

def assign_cluster_positions(clusters, page_width, page_height):
    """
    For each cluster, compute a textual position label from its bounding box.
    clusters: dict of { cluster_id: {"boundingBox": [...], ...}, ... }
    page_width, page_height: page dimensions
    """
    for cid, cinfo in clusters.items():
        bbox = cinfo.get("boundingBox", None)
        if bbox:
            position_label = classify_position(bbox, page_width, page_height)
            cinfo["position"] = position_label
        else:
            cinfo["position"] = "unknown"


In [28]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Chris Sweet, Priscila Moreira
Date Created: February 24, 2025
Description: Extracts key concepts from chunk using enriched data and chunk 
             image.
--------------------------------------------------------------------------------
"""

# grap sub-image
def crop_with_margin(image, bbox, margin=10):
    """
    Crop the image using the provided bounding box with an added margin.
    
    Parameters:
      image: The main image as a NumPy array.
      bbox: List or tuple in the form [x_min, y_min, x_max, y_max].
      margin: Number of pixels to add as a margin around the bbox.
    
    Returns:
      The cropped image as a NumPy array.
    """
    x_min, y_min, x_max, y_max = bbox
    h, w = image.shape[:2]
    
    # Expand the bounding box by margin and clip to image dimensions.
    x_min = max(0, int(x_min) - margin)
    y_min = max(0, int(y_min) - margin)
    x_max = min(w, int(x_max) + margin)
    y_max = min(h, int(y_max) + margin)
    
    return image[y_min:y_max, x_min:x_max]

def numpy_image_to_base64_png(numpy_image):
    """
    Convert a cropped image in NumPy array format (BGR or RGB)
    to a base64-encoded PNG string.
    """
    # If you used cv2.imread or manipulations, you likely have BGR. 
    # If it's already RGB, you might not need conversion. 
    # Typically, no big difference for encoding as PNG.
    
    # Encode image to PNG in memory
    success, encoded_buffer = cv2.imencode(".png", numpy_image)
    if not success:
        raise ValueError("Could not encode numpy image as PNG.")
    
    # Convert to base64 text
    b64_str = base64.b64encode(encoded_buffer).decode("utf-8")
    return b64_str

def build_chunk_prompt(chunk):
    """
    Build a prompt that includes all relevant chunk data, plus a base64-encoded image,
    asking the LLM to extract key concepts.
    
    chunk: a DocumentChunk object or dict with fields:
      - type
      - summary
      - description
      - key_value_pairs (list or dict)
      - mainText (the chunk text)
    Image: The cropped image for this chunk
    
    Returns: A string prompt ready for the LLM.
    """
    
    # Convert key_value_pairs to JSON or a string representation
    kv_pairs_str = json.dumps(chunk.key_value_pairs, indent=2) if chunk.key_value_pairs else "None"
    
    prompt = f"""<context>
Chunk Type: {chunk.type}
Chunk Summary: {chunk.summary}
Chunk Description: {chunk.description}
Key-Value Pairs: {kv_pairs_str}
Main Text:
{chunk.mainText}
</context>

You are an expert in document layout, semantic analysis, and ontology. 
We have included an image referenced above (not embedded here), which you can consult if your environment supports image retrieval.
Please extract key concepts from this chunk's text and image. 
Focus on domain-specific items like dates, amounts, roles, references, etc.

Return ONLY a JSON object of this form:
{{
  "concepts": [
    {{
      "type": "string, e.g., date, cost, entity",
      "value": "string with recognized concept"
    }}
  ],
  "notes": "Any additional observations about the chunk or the image."
}}
"""
    return prompt


In [46]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Charles Vardeman II, Chris Sweet
Date Created: February 23, 2025
Description: Pydantic definitions for DocumentChunk and DocumentGraph,
             Enrichment prompts and conversion from Clusters to DocumentChunk.
--------------------------------------------------------------------------------
## KAG Graph Model
Core Built-in Properties (ptb):
1. supporting_chunks: Links that connect instances to their source text chunks in the original documents. This enables traceability between the knowledge graph and source materials.

2. description: Contains general descriptive information that can be attached at two levels:
   - When attached to a type (tk): Provides global description for that type
   - When attached to an instance (ei): Provides instance-specific descriptive information consistent with the original document context

3. summary: A condensed representation of either:
   - An instance (ei)
   - A relation (rj)

4. belongTo: Represents the inductive relationship that connects instances to their concepts in the concept hierarchy

Chunk Entity Properties:
1. id: A composite identifier made up of:
   - articleID: The globally unique article identifier
   - paraCode: The paragraph code within the article
   - idInPara: The sequential code of the chunk within the paragraph
   These are concatenated with "#" as the connector

2. mainText: The actual content of the chunk

3. summary: A summary of the chunk's content

The model also supports two categories of additional properties:

- ptc (Static/Knowledge Area):
  - Pre-defined properties by domain experts
  - Subject to schema constraints
  - Used primarily for professional decision-making applications

- ptf (Dynamic/Information Area):
  - Properties added in an ad-hoc manner
  - More flexible and schema-free
  - Used primarily for information retrieval applications

Both ptc and ptf share the same conceptual terminology and can coexist in the instance storage space, allowing applications to balance between rigorous knowledge representation and flexible information handling based on their specific needs.

This property structure enables KAG to maintain clear connections between formal knowledge structures and original document contexts while supporting both strict domain knowledge and flexible information extraction.

"""

def mk_did(prefix, txt): return f"did:{prefix}:{sha256(txt.encode()).hexdigest()[:16]}"

# Chunk, Graph classes ##########################################################
class DocumentChunk(BaseModel):
    id: str  # KAG format: articleID#paraCode#idInPara
    mainText: str
    summary: str
    description: str
    type: str  # Keep for now, will evolve in KGfr
    region_id: str  # Keep for layout tracking
    relationships: List[Dict]
    position: str
    boundingBox: List[float]
    key_value_pairs: List[Dict]
    concepts: Dict
    # KAG required properties
    supporting_chunks: List[str] = field(default_factory=list)
    belongTo: str  # Basic concept for now

DOCUMENT_CONTEXT = {
    "@vocab": "http://schema.org/",
    "doc": "http://example.org/document#",
    "chunk": "http://example.org/chunk#",
    "id": "@id",
    "type": "@type",
    "relationships": {
        "@id": "chunk:hasRelation",
        "@type": "@id"
    }
}

class DocumentGraph(BaseModel):
    id: str
    document_type: str
    chunks: List[DocumentChunk]
    metadata: Dict
    context: Dict = DOCUMENT_CONTEXT

# Enrichment prompts ############################################################
SUMMARY_PROMPT = """Create a concise summary of this text chunk from a {doc_type} document.
The chunk type is: {chunk_type}

Text to summarize:
{text}

Return ONLY the summary text, no additional explanation."""

DESCRIPTION_PROMPT = """Create a detailed description of this document chunk that explains its role and context.
Document type: {doc_type}
Chunk type: {chunk_type}
Position: {position}

Content:
{text}

Return ONLY the description, no additional explanation."""

CONCEPT_PROMPT = """Identify the most specific concept type that this chunk belongs to.
Use standard document ontology concepts (e.g., 'Confidentiality Notice', 'Contact Information', 'Document Header').

Content type: {chunk_type}
Text:
{text}

Return ONLY the concept name, no additional explanation."""

# Enrichment LLM calls ##########################################################
def enrich_chunk(chunk: dict, doc_type: str, chat=None) -> dict:
    "Enrich a chunk with KAG properties using LLM"
    chat = chat or Chat(model)
    
    # Generate summary
    summary = chat(SUMMARY_PROMPT.format(
        doc_type=doc_type,
        chunk_type=chunk.type,
        text=chunk.mainText
    )).choices[0].message.content.strip()
    
    # Generate description
    description = chat(DESCRIPTION_PROMPT.format(
        doc_type=doc_type,
        chunk_type=chunk.type,
        position=chunk.position,
        text=chunk.mainText
    )).choices[0].message.content.strip()
    
    # Identify concept
    concept = chat(CONCEPT_PROMPT.format(
        chunk_type=chunk.type,
        text=chunk.mainText
    )).choices[0].message.content.strip()
    
    return DocumentChunk(
        id=chunk.id,
        type=chunk.type,
        mainText=chunk.mainText,
        summary=summary,
        description=description,
        belongTo=concept,
        region_id=chunk.region_id,
        relationships=chunk.relationships,
        position=chunk.position,
        boundingBox=chunk.boundingBox,
        key_value_pairs=chunk.key_value_pairs,
        concepts={}
    ) #.model_dump()

# Cluster to Chunk conversion ###################################################
# TODO: Add paragraphs, bounding box, key-concepts, key-value pairs 
def cluster_to_document_chunk(cluster):
    """
    Convert a cluster dictionary into a DocumentChunk object.
    Expects the cluster dict to have at least these keys:
      - "id"
      - "type"
      - "content" (the main text)
      - "region_id"
      - "relationships" (a list or empty list)
    Optionally, you can supply or generate summary, description, and belongTo (concept).
    """

    return DocumentChunk(
        id=mk_did('chunk', cluster['text']),
        type=cluster["type"],
        mainText=cluster["text"],
        summary="",
        description="",
        belongTo="",
        region_id=cluster["name"],
        relationships=cluster.get("relationships", []),
        position=cluster.get("position", ""),
        boundingBox=cluster.get("boundingBox", []),
        key_value_pairs=cluster.get("key_value_pairs", []),
        concepts={}
    )

def convert_clusters_to_document_chunks(clusters):
    """
    Convert a dictionary of clusters into a list of DocumentChunk objects.
    
    Parameters:
      clusters (dict): A dictionary of form { <cid>: {...}, ... }
    
    Returns:
      list[DocumentChunk]
    """
    document_chunks = []
    for cid, cluster_dict in clusters.items():
        doc_chunk = cluster_to_document_chunk(cluster_dict)
        document_chunks.append(doc_chunk)
    return document_chunks

def parse_json(s: str) -> dict:
    "Parse JSON from raw LLM response, handling markdown code blocks"
    if isinstance(s, (dict,list)): return s
    if hasattr(s, 'choices'): s = s.choices[0].message.content
    
    # Extract JSON from markdown code block if present
    if '```' in s:
        blocks = s.split('```')
        # Take the content between ```json and ```
        for i, block in enumerate(blocks):
            if block.strip().lower() == 'json' and i+1 < len(blocks):
                s = blocks[i+1]
                break
    
    # Clean up and parse
    s = s.strip()
    if not s.startswith('{'): s = s[s.find('{'):]
    if not s.endswith('}'): s = s[:s.rfind('}')+1]
    return json.loads(s)

def mk_meta(img_name, doc_type, sorted_clusters):
    "Create document metadata"
    
    # get reading ordered names
    sc_names = [cluster_dict["name"] for (cid, cluster_dict) in sorted_clusters]

    return {
        'source': img_name,
        'created': datetime.now().isoformat(),
        'document_type': doc_type,
        'num_regions': len(sc_names),
        'analysis_version': '1.1',
        'region_ids': sc_names,
        'content_types': list(set(cluster_dict["type"] for (cid, cluster_dict) in sorted_clusters))
    }


In [51]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Chris Sweet, Priscila Moreira
Date Created: February 23, 2025
Description:  Function to extract Document Graph from the Layout
--------------------------------------------------------------------------------
"""

def create_graph(ocr_data, image, img_name):
    # Step 1. Make key/pairs/tables atomic by merged paragraphs ####################
    # Step 1a: Load OCR/Layout Dict. ###############################################
    # input_path = '../datasets/FUNSD/training_data/annotations_new/0001463282_prebuilt_layout.json'   # Update with your OCR JSON file path

    # with open(input_path, "r", encoding="utf-8") as f:
    #     ocr_data = json.load(f)

    # Step 1b: Update the Dict. with the key/pair merged paragraphs ################
    updated_kp_ocr = merge_key_value_pairs_create_new_paragraphs(ocr_data, iou_threshold=0.1)

    # Step 1b: Update the Dict. with the key/pair merged paragraphs ################
    updated_kp_table_ocr = merge_table_to_paragraph(updated_kp_ocr)

    # Step 2. Generate clusters ####################################################
    # Step 2a. Generate cluster labels #############################################
    cluster_labels = generate_clusters(updated_kp_table_ocr)

    if not cluster_labels:
        print("No paragraphs found in clustering! Stopping Execution.")
        return None

    # Step 2b. Get cluster text and bounding box ####################################

    # Group and sort paragraphs by cluster
    clusters = group_paragraphs_by_cluster(updated_kp_table_ocr, cluster_labels)

    # Step 2c. Get cluster relationships ############################################
    rels = compute_cluster_relationships(clusters, margin=400)
    relationships = transform_relationships(rels, clusters)

    # put relationships into cluster
    for idx, cluster in clusters.items():
        cluster["relationships"] = relationships.get(idx, [])

    # Step 2d. Get cluster reading order ############################################
    sorted_clusters = sort_clusters_reading_order(clusters)

    # Step 2e. Get cluster position description #####################################
    # TODO: Need to handle multiple pages, start with page 0
    width = int(updated_kp_table_ocr["pages"][0]["width"])
    height = int(updated_kp_table_ocr["pages"][0]["height"])
    assign_cluster_positions(clusters, page_width=width, page_height=height)
        
    # for cid, cinfo in clusters.items():
    #     print(f"Cluster {cid} => position: {cinfo['position']}")

    # Step 3. Enrich clusters #######################################################
    # Set LLM model
    model = models[2]

    # Setup DocumentChunks
    document_chunks = convert_clusters_to_document_chunks(clusters)

    # Enrich with summary, description, concept
    enriched_chunks = []

    for chunk in document_chunks:
        enriched_chunk = enrich_chunk(
                            chunk=chunk,
                            doc_type="Unknown", # we find this via our key concept similarity
                            )
        enriched_chunks.append(enriched_chunk)

    # Step 4. Extract Key Concepts ##################################################
    # Load your main image
    # image_path = "../datasets/FUNSD/training_data/images/0001463282.png"  # Replace with your image path
    # image = cv2.imread(image_path)
    chat = Chat(model, sp="""You are a helpful and concise assistant with ontology experience.""")

    for chunk in enriched_chunks:
        # crop image
        cropped_image = crop_with_margin(image, chunk.boundingBox)

        # convert cropped image
        #cropped_image_b64 = numpy_image_to_base64_png(cropped_image)
        cv2.imwrite("cropped_image_tmp.png", cropped_image)
        with open("cropped_image_tmp.png", "rb") as f:
            cropped_image_b64 = f.read()

        # generate prompt
        prompt = build_chunk_prompt(chunk)

        # query LLM
        key_concepts = parse_json(chat([cropped_image_b64, prompt]).choices[0].message.content.strip())

        # save concepts
        chunk.concepts = key_concepts
        #print(key_concepts)
        
    # Step 5. Create document graph #################################################
    """
    "metadata": {
        "source": "82092117.png",
        "created": "2025-02-22T16:53:01.911217",
        "document_type": "fax cover sheet",
        "num_regions": 7,
        "analysis_version": "1.0",
        "region_ids": [
            "r1",
            "r2",
            "r3",
            "r4",
            "r5",
            "r6",
            "r7"
        ],
        "content_types": [
            "contact information",
            "header",
            "instructions",
            "notice",
            "note",
            "footer"
        ]
    },
    """
    return DocumentGraph(
        id = mk_did('doc', img_name),
        document_type = "Unknown",
        chunks = [chunk.model_dump() for chunk in enriched_chunks],
        metadata = mk_meta(img_name, "Unknown", sorted_clusters)
    ).model_dump()

In [49]:
"""
--------------------------------------------------------------------------------
File Name:    layout_extraction.ipynb
Authors:      Chris Sweet, Priscila Moreira
Date Created: February 23, 2025
Description:  Main calling function.
--------------------------------------------------------------------------------
"""
# Load OCR/Layout file ##########################################################
input_path = '../datasets/FUNSD/training_data/annotations_new/0001463282_prebuilt_layout.json'   # Update with your OCR JSON file path

with open(input_path, "r", encoding="utf-8") as f:
    ocr_data = json.load(f)

# Load document image file ######################################################
image_name = "0001463282.png"
image_path = "../datasets/FUNSD/training_data/images/0001463282.png"  # Replace with your image path
image = cv2.imread(image_path)

# Extract Document graph ########################################################
document_graph = create_graph(ocr_data, image, image_name)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [50]:
document_graph

DocumentGraph(id='did:doc:0a2d266b78086cba', document_type='Unknown', chunks=[DocumentChunk(id='did:chunk:b27814894a394d3c', mainText='MARKETING RESEARCH AUTHORIZATIU .. (Recommended Proposal Attached)', summary='Marketing Research Authorization (Recommended Proposal Attached)', description="This document chunk serves as the title for a document found at the top of the page, indicating the subject and purpose of the content that follows. It refers to a 'Marketing Research Authorization,' suggesting that the document outlines approval or permission for marketing research activities. The phrase 'Recommended Proposal Attached' implies that a specific proposal related to the marketing research is included with the document, potentially providing detailed plans or recommendations for conducting the research. The misspelling 'AUTHORIZATIU' may indicate a typographical error, but it doesn't change the intended meaning of the document title. This title sets the stage for the reader by clearly 